# Lab 2.3: Instructions vs. Permissions and Hooks

**Day 1 - Topic 2**

Goal: show that CLAUDE.md is advice, while a PreToolUse hook can block a tool call before it runs and a PostToolUse hook can only report after the write.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab2_3_instructions_vs_permissions_hooks/start/lab2_3_instructions_vs_permissions_hooks.ipynb)

In [ ]:
# No extra packages needed: this lab uses the Python standard library only.
# If running locally with the course venv already activated, you can skip this cell.

In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/context-claude-companion.git"


def find_repo_root() -> Path:
    """Use the local companion checkout if the notebook runs inside it; otherwise clone it (Colab)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "day1" / "demos" / "fixtures").is_dir():
            return p
    clone = Path("context-claude-companion").resolve()
    if not clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return clone


REPO_ROOT = find_repo_root()
FIXTURES = REPO_ROOT / "day1" / "demos" / "fixtures"
print(f"Using fixtures in {FIXTURES}")


def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: Three layers of control

The fixture repo has advisory prose in `CLAUDE.md`, `permissions.deny` rules that Claude Code enforces itself, and two hook scripts. This cell lists all three. The deny rules are only listed here, not evaluated.

In [ ]:
FIXTURE = FIXTURES / "hooks_repo"
settings = json.loads((FIXTURE / ".claude" / "settings.json").read_text())


def sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:8]


print("CLAUDE.md (advice the model may or may not follow):")
for line in (FIXTURE / "CLAUDE.md").read_text().splitlines():
    if line.startswith("- "):
        print(f"  {line}")

print("\npermissions.deny in settings.json (enforced by Claude Code, listed here, not evaluated):")
for rule in settings["permissions"]["deny"]:
    print(f"  {rule}")

print("\nHooks registered in settings.json:")
for event, entries in settings["hooks"].items():
    for entry in entries:
        print(f"  {event:<12} matcher={entry['matcher']:<14} -> {entry['hooks'][0]['command']}")

## Section 2: Find the hooks for a tool call

Each hook entry has a `matcher` regex. A hook applies when the matcher matches the whole tool name.

In [ ]:
def hooks_for(settings: dict, event: str, tool_name: str) -> list[str]:
    """Return hook commands whose matcher regex matches the tool name."""
    commands = []
    # TODO: Loop over the entries in settings.get("hooks", {}).get(event, [])
    # TODO: Keep an entry only when re.fullmatch(entry.get("matcher", ".*"), tool_name) is truthy
    # TODO: For a kept entry, add h["command"] to commands for every hook h in entry["hooks"] whose h["type"] == "command"
    return commands

In [ ]:
print(hooks_for(settings, "PreToolUse", "Edit"))
print(hooks_for(settings, "PostToolUse", "Bash"))
# Expected: ['python3 .claude/hooks/pre_tool_use.py'], then []
check("Edit triggers the PreToolUse hook", hooks_for(settings, "PreToolUse", "Edit") == ["python3 .claude/hooks/pre_tool_use.py"])
check("Bash triggers no PostToolUse hook", hooks_for(settings, "PostToolUse", "Bash") == [])

## Section 3: Run a hook script

Claude Code starts the hook command, writes the tool call as JSON to its stdin, and reads the exit code. Exit code 2 blocks the call and sends stderr back to Claude.

In [ ]:
def run_hook(command: str, root: Path, event: str, tool_name: str, tool_input: dict) -> tuple[int, str]:
    """Run one hook script the way Claude Code does and return (exit code, stderr)."""
    payload = {"hook_event_name": event, "tool_name": tool_name, "tool_input": tool_input}
    # TODO: Run command with subprocess.run(command, shell=True, cwd=root, capture_output=True, text=True, input=...)
    # TODO: Pass json.dumps(payload) as the input= argument: the hook protocol sends the tool call as JSON on stdin
    # TODO: Return (proc.returncode, proc.stderr.strip()). Exit code 2 means "block the tool call" and stderr goes back to Claude
    return 0, ""

## Section 4: Run four scenarios

`run_scenario` is provided. It runs the `PreToolUse` hooks, performs the write if nothing blocked it, runs the `PostToolUse` hooks, and compares the file hash before and after. It works on a temporary copy of the fixture.

In [ ]:
def run_scenario(root: Path, settings: dict, title: str, tool_name: str, tool_input: dict, new_content: str = "") -> tuple[bool, bool]:
    """Run PreToolUse hooks, the write if allowed, then PostToolUse hooks. Return (blocked, file_changed)."""
    target = root / tool_input["file_path"] if "file_path" in tool_input else None
    before = sha(target) if target else "-"
    # Claude Code sends absolute paths; mirror that.
    sent = dict(tool_input, file_path=str(target)) if target else dict(tool_input)

    print(f"--- {title} ---")
    print(f"  Tool call      : {tool_name} {tool_input.get('file_path') or tool_input.get('command')}")

    blocked = False
    for cmd in hooks_for(settings, "PreToolUse", tool_name):
        code, err = run_hook(cmd, root, "PreToolUse", tool_name, sent)
        print(f"  PreToolUse     : `{cmd}` -> exit {code}" + (f", stderr: {err}" if err else ""))
        blocked = blocked or code == 2
    if blocked:
        print("  Tool executed  : NO (blocked before any side effect)")
        changed = bool(target) and sha(target) != before
        if target:
            print(f"  File on disk   : {before} -> {sha(target)} ({'CHANGED' if changed else 'unchanged'})")
        print()
        return True, changed

    if target:
        target.write_text(new_content)  # stand-in for the Edit tool performing the write
    print("  Tool executed  : YES")
    for cmd in hooks_for(settings, "PostToolUse", tool_name):
        code, err = run_hook(cmd, root, "PostToolUse", tool_name, sent)
        print(f"  PostToolUse    : `{cmd}` -> exit {code}" + (f", stderr: {err}" if err else ""))
    changed = bool(target) and sha(target) != before
    if target:
        print(f"  File on disk   : {before} -> {sha(target)} ({'CHANGED, write already happened' if changed else 'unchanged'})")
    print()
    return False, changed

In [ ]:
workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name).resolve() / "hooks_repo"
shutil.copytree(FIXTURE, root)

r1 = run_scenario(root, settings, "1. Edit a locked migration", "Edit",
                  {"file_path": "migrations/locked/0001_init.sql"},
                  "ALTER TABLE orders DROP COLUMN audit_id;")
r2 = run_scenario(root, settings, "2. Edit pricing with a float conversion", "Edit",
                  {"file_path": "services/orders/pricing.py"},
                  "def apply_discount(amount, bps):\n    return float(amount) * 0.10\n")
r3 = run_scenario(root, settings, "3. Edit pricing with integer cents", "Edit",
                  {"file_path": "services/orders/pricing.py"},
                  "def apply_discount(amount_cents, bps):\n    return (amount_cents * bps) // 10_000\n")
r4 = run_scenario(root, settings, "4. Recursive delete", "Bash", {"command": "rm -rf migrations"})
workdir.cleanup()

# Expected: scenarios 1 and 4 are blocked (exit 2) before any side effect.
# Scenario 2 is NOT blocked: the PostToolUse hook reports the float only after the write.
check("Scenario 1: PreToolUse blocks the locked migration and the file is unchanged", r1 == (True, False))
check("Scenario 2: the float edit is allowed and the write already happened", r2 == (False, True))
check("Scenario 3: the integer-cents edit is allowed", r3 == (False, True))
check("Scenario 4: the recursive delete is blocked", r4[0] is True)

In [ ]:
print("Lab complete.")
print("Takeaway: CLAUDE.md steers, a PreToolUse hook stops the call before it runs (exit 2), and a PostToolUse hook only reports after the write. Back both with OS permissions and CI.")